# Phase 3 — Deep Learning
**Models:** Dense Neural Network · 1D Convolutional Neural Network  
**Goal:** Train two neural network architectures and compare their performance against the classical ML baselines.

## 1. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)
from imblearn.over_sampling import SMOTE
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
import joblib, os, warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
tf.random.set_seed(42)

DATA_PATH   = r"C:\Users\hp\Desktop\Projets\cyber-ids-project\data"
MODELS_PATH = r"C:\Users\hp\Desktop\Projets\cyber-ids-project\models"
PLOTS_PATH  = r"C:\Users\hp\Desktop\Projets\cyber-ids-project\reports"
os.makedirs(MODELS_PATH, exist_ok=True)

## 2. Load Data & Prepare

In [ ]:
df = pd.read_parquet(os.path.join(DATA_PATH, 'clean_data.parquet'))
le = joblib.load(os.path.join(DATA_PATH, 'label_encoder.pkl'))

X = df.drop('Label', axis=1)
y = df['Label']
n_features = X.shape[1]
n_classes  = len(le.classes_)
print(f"✅ {df.shape[0]:,} rows  ·  {n_features} features  ·  {n_classes} classes")

## 3. Train / Test Split & SMOTE

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

unique, counts = np.unique(y_train, return_counts=True)
sampling_dict  = {u: 10000 for u, c in zip(unique, counts) if c < 10000}
smote = SMOTE(sampling_strategy=sampling_dict, random_state=42, k_neighbors=5)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
print(f"Training set after SMOTE: {X_train_sm.shape[0]:,} rows")

X_train_np = np.array(X_train_sm);  X_test_np  = np.array(X_test)
y_train_np = np.array(y_train_sm);  y_test_np  = np.array(y_test)

y_train_cat = keras.utils.to_categorical(y_train_np, n_classes)
y_test_cat  = keras.utils.to_categorical(y_test_np,  n_classes)

## 4. Evaluation & Plotting Helpers

In [ ]:
def evaluate_dl_model(name, model, X_test, y_test_np, le):
    y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
    acc  = accuracy_score(y_test_np, y_pred)
    prec = precision_score(y_test_np, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_test_np, y_pred, average='weighted', zero_division=0)
    f1   = f1_score(y_test_np, y_pred, average='weighted', zero_division=0)

    print(f"{'='*50}\n  {name}\n{'='*50}")
    print(f"  Accuracy  : {acc:.4f} ({acc*100:.2f}%)")
    print(f"  Precision : {prec:.4f}  |  Recall : {rec:.4f}  |  F1 : {f1:.4f}")
    print(classification_report(y_test_np, y_pred, target_names=le.classes_, zero_division=0))

    cm = confusion_matrix(y_test_np, y_pred)
    plt.figure(figsize=(14, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
                xticklabels=le.classes_, yticklabels=le.classes_)
    plt.title(f'Confusion Matrix — {name}', fontsize=14, fontweight='bold')
    plt.ylabel('True Label'); plt.xlabel('Predicted Label')
    plt.xticks(rotation=45, ha='right', fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_PATH, f'cm_{name.replace(" ", "_")}.png'), dpi=150)
    plt.show()
    return {'name': name, 'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1}

def plot_history(history, name):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    for ax, metric, ylabel in [(ax1, 'accuracy', 'Accuracy'), (ax2, 'loss', 'Loss')]:
        ax.plot(history.history[metric],     label='Train',      color='#3498db')
        ax.plot(history.history[f'val_{metric}'], label='Validation', color='#e74c3c')
        ax.set_title(f'{name} — {ylabel}', fontweight='bold')
        ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
        ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_PATH, f'history_{name.replace(" ", "_")}.png'), dpi=150)
    plt.show()

## 5. Model 1 — Dense Neural Network

**Architecture:** Input(66) → Dense(256) → BN → Dropout → Dense(128) → BN → Dropout → Dense(64) → BN → Dropout → Output(15, softmax)

In [ ]:
def build_dense_model(n_features, n_classes):
    return keras.Sequential([
        layers.Input(shape=(n_features,)),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(), layers.Dropout(0.3),
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(), layers.Dropout(0.3),
        layers.Dense(64, activation='relu'),
        layers.BatchNormalization(), layers.Dropout(0.2),
        layers.Dense(n_classes, activation='softmax')
    ], name='dense_nn')

dense_model = build_dense_model(n_features, n_classes)
dense_model.compile(optimizer=keras.optimizers.Adam(1e-3),
                    loss='categorical_crossentropy', metrics=['accuracy'])
dense_model.summary()

In [ ]:
cb_dense = [
    callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)
]

dense_history = dense_model.fit(
    X_train_np, y_train_cat,
    epochs=30, batch_size=1024, validation_split=0.1,
    callbacks=cb_dense, verbose=1)

results_dense = evaluate_dl_model("Dense Neural Network", dense_model, X_test_np, y_test_np, le)
plot_history(dense_history, "Dense Neural Network")
dense_model.save(os.path.join(MODELS_PATH, 'dense_model.keras'))
print("✅ Dense model saved")

## 6. Model 2 — 1D Convolutional Neural Network

**Architecture:** Input(66,1) → Conv1D(64) → MaxPool → Conv1D(128) → MaxPool → Conv1D(256) → GlobalMaxPool → Dense(128) → Dense(64) → Output(15, softmax)

In [ ]:
X_train_cnn = X_train_np.reshape(X_train_np.shape[0], n_features, 1)
X_test_cnn  = X_test_np.reshape(X_test_np.shape[0],  n_features, 1)

def build_cnn_model(n_features, n_classes):
    return keras.Sequential([
        layers.Input(shape=(n_features, 1)),
        layers.Conv1D(64,  kernel_size=3, activation='relu', padding='same'),
        layers.BatchNormalization(), layers.MaxPooling1D(2), layers.Dropout(0.2),
        layers.Conv1D(128, kernel_size=3, activation='relu', padding='same'),
        layers.BatchNormalization(), layers.MaxPooling1D(2), layers.Dropout(0.2),
        layers.Conv1D(256, kernel_size=3, activation='relu', padding='same'),
        layers.BatchNormalization(), layers.Dropout(0.2),
        layers.GlobalMaxPooling1D(),
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(), layers.Dropout(0.3),
        layers.Dense(64, activation='relu'), layers.Dropout(0.2),
        layers.Dense(n_classes, activation='softmax')
    ], name='cnn_1d')

cnn_model = build_cnn_model(n_features, n_classes)
cnn_model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss='categorical_crossentropy', metrics=['accuracy'])
cnn_model.summary()

In [ ]:
cb_cnn = [
    callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6),
    callbacks.ModelCheckpoint(os.path.join(MODELS_PATH, 'cnn_model_checkpoint.keras'),
                              monitor='val_accuracy', save_best_only=True, verbose=1)
]

cnn_history = cnn_model.fit(
    X_train_cnn, y_train_cat,
    epochs=30, batch_size=1024, validation_split=0.1,
    callbacks=cb_cnn, verbose=1)

results_cnn = evaluate_dl_model("1D-CNN", cnn_model, X_test_cnn, y_test_np, le)
plot_history(cnn_history, "1D-CNN")
cnn_model.save(os.path.join(MODELS_PATH, 'cnn_model.keras'))
print("✅ 1D-CNN model saved")